In [75]:
# import Pkg; Pkg.add("Pipe")
# import Pkg; Pkg.add("HypothesisTests")
# import Pkg; Pkg.add("StatsPlots")
# import Pkg; Pkg.add("Combinatorics")
# import Pkg; Pkg.add("StatsModels")
# import Pkg; Pkg.add("MLJ")
# import Pkg; Pkg.add("MLJLinearModels")
# import Pkg; Pkg.add("MLJDecisionTreeInterface")
# import Pkg; Pkg.add("Flux")

In [76]:
# Projet de Prédiction de Consommation en Carburant
# MTH3302 - Modèle de Machine Learning

# Importation des bibliothèques nécessaires
using CSV
using DataFrames
using Random
using MLJ
using MLJModels
using MLJDecisionTreeInterface
using Flux
using StatsBase
using Plots

In [79]:
# Charger et prétraiter les données

# Charger les données
train_df = CSV.read("train.csv", DataFrame, delim=';', decimal=',')
test_df = CSV.read("test.csv", DataFrame, delim=';', decimal=',')

function remove_outliers(df, col)
    q1 = quantile(df[!, col], 0.25)
    q3 = quantile(df[!, col], 0.75)
    iqr = q3 - q1
    lower_bound = q1 - 1.5 * iqr
    upper_bound = q3 + 1.5 * iqr
    println("Retrait des valeurs aberrantes pour $col: [$lower_bound, $upper_bound]")
    return df[(df[!, col] .> lower_bound) .& (df[!, col] .< upper_bound), :]
end

# Visualisation de la consommation en fonction de toutes les variables
for col in names(train_df)
    if eltype(train_df[!, col]) <: Number
        train_df = remove_outliers(train_df, col)
        # plot_consumption(train_df, col)
    end
end

# Prétraitement des données catégorielles
categorical_cols = [:type, :transmission, :boite]

# Encodage one-hot pour les variables catégorielles
function one_hot_encode(data, cat_cols)
    # Coercition des types catégoriels
    for feature in cat_cols
        if eltype(data[:, feature]) <: AbstractString
            coerce!(data, feature => Multiclass)
        end
    end

    # Encoder avec suppression de la dernière colonne pour éviter la multicolinéarité
    one_hot = machine(OneHotEncoder(drop_last=true), select(data, cat_cols))
    fit!(one_hot, verbosity=0)
    
    # Transformer uniquement les colonnes catégorielles
    encoded_cats = MLJ.transform(one_hot, select(data, cat_cols))
    
    # Supprimer les colonnes catégorielles originales et ajouter les colonnes encodées
    select!(data, Not(cat_cols))
    return hcat(data, encoded_cats)
end

train_df = one_hot_encode(train_df, categorical_cols)
test_df = one_hot_encode(test_df, categorical_cols)

Retrait des valeurs aberrantes pour annee: [2007.0, 2031.0]
Retrait des valeurs aberrantes pour nombre_cylindres: [1.0, 9.0]
Retrait des valeurs aberrantes pour cylindree: [0.050000000000000266, 5.25]
Retrait des valeurs aberrantes pour consommation: [4.137953703703698, 16.33402777777778]


Row,annee,nombre_cylindres,cylindree,type__VUS_petit,type__VUS_standard,type__break_moyen,type__break_petit,type__camionnette_petit,type__camionnette_standard,type__monospace,type__voiture_compacte,type__voiture_deux_places,type__voiture_grande,type__voiture_minicompacte,type__voiture_moyenne,transmission__4x4,transmission__integrale,transmission__propulsion,boite__automatique
,Int64,Int64,Float64,Float64,Float64,Float64,Float64,Float64,Float64,Float64,Float64,Float64,Float64,Float64,Float64,Float64,Float64,Float64,Float64
1,2014,4,2.5,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0
2,2014,4,2.5,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0
3,2014,4,2.5,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
4,2014,4,2.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0
5,2014,8,5.8,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0
6,2014,8,5.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0
7,2014,8,5.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0
8,2014,4,2.4,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0
9,2014,6,3.5,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0


In [80]:
using Statistics  # Pour mean et std
using DataFrames  # Assurez-vous que DataFrames est installé et chargé

# Fonction pour standardiser les colonnes sélectionnées
function standardize_columns!(df::DataFrame, cols::Vector{Symbol})
    for col in cols
        μ = mean(df[!, col])
        σ = std(df[!, col])
        df[!, col] = (df[!, col] .- μ) ./ σ
    end
    return df
end

# Normalisation des caractéristiques numériques
numeric_cols = [:annee, :nombre_cylindres, :cylindree]

# Standardisation des données (transformation en place)
train_df = standardize_columns!(train_df, numeric_cols)
test_df = standardize_columns!(test_df, numeric_cols)

Row,annee,nombre_cylindres,cylindree,type__VUS_petit,type__VUS_standard,type__break_moyen,type__break_petit,type__camionnette_petit,type__camionnette_standard,type__monospace,type__voiture_compacte,type__voiture_deux_places,type__voiture_grande,type__voiture_minicompacte,type__voiture_moyenne,transmission__4x4,transmission__integrale,transmission__propulsion,boite__automatique
,Float64,Float64,Float64,Float64,Float64,Float64,Float64,Float64,Float64,Float64,Float64,Float64,Float64,Float64,Float64,Float64,Float64,Float64,Float64
1,-1.53728,-0.769722,-0.372358,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0
2,-1.53728,-0.769722,-0.372358,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0
3,-1.53728,-0.769722,-0.372358,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
4,-1.53728,-0.769722,-0.739816,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0
5,-1.53728,1.45063,2.05287,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0
6,-1.53728,1.45063,1.46493,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0
7,-1.53728,1.45063,1.46493,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0
8,-1.53728,-0.769722,-0.44585,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0
9,-1.53728,0.340454,0.362559,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0


In [81]:
# Diviser les données d'entraînement

target_col = :consommation
split_ratio = 0.8
# Définir la graine pour la reproductibilité
Random.seed!(42)

# Calculer l'index de séparation
n = nrow(train_df)
train_idx = randperm(n)[1:floor(Int, n * split_ratio)]
test_idx = setdiff(1:n, train_idx)

# Séparer les données
X = select(train_df, Not(target_col))
y = train_df[!, target_col]

X_train = X[train_idx, :]
y_train = y[train_idx]
X_test = X[test_idx, :]
y_test = y[test_idx]

75-element Vector{Float64}:
  9.80041666666667
 14.700625
 11.2004761904762
  9.4084
 12.3794736842105
  7.3503125
  9.80041666666667
 10.2265217391304
  9.80041666666667
  7.58741935483871
  9.4084
  7.84033333333333
 10.2265217391304
  ⋮
 10.6913636363636
 11.7605
  9.80041666666667
 11.7605
  6.91794117647059
  7.58741935483871
  7.3503125
  8.40035714285714
  9.04653846153846
 11.7605
 10.2265217391304
 12.3794736842105

In [84]:
include("Jeremie_Utils.jl")
m1_train = deepcopy(X_train)
m1_valid = deepcopy(y_train)
# Set features to be all column names of m1_train except consommation
features = names(m1_train)
# add m1_valid to m1_train with column name consommation
m1_train = hcat(m1_train, DataFrame(consommation=m1_valid))
# Convert features from String to Symbol
features = Symbol.(features)
m₁ = create_model(m1_train, features, :consommation)

ŷ = StatsModels.predict(m₁, X_train)
y_valid = m1_train[:, :consommation]

rms(ŷ, y_valid)

0.8562391937641404

In [86]:
id = 1:size(test_df, 1)

ŷ = StatsModels.predict(m₁, test_df)

df_pred = DataFrame(id=id, consommation=ŷ)

CSV.write("benchmark.csv", df_pred)

"benchmark.csv"